# 16 — Calibrate → Integrate Handoff

Calibration is a *means*, not an end. After `calibrate()` writes the refined geometry + residual-correction map, downstream tools have to consume both. This notebook closes that loop:

1. Run `midas_calibrate_v2.calibrate(image, wavelength, …)` on a CeO₂ image.
2. Write the v1-compatible binary `residual_corr.bin` + a paramstest-style geometry file.
3. Load both into `midas_integrate_v2`'s `IntegrationSpec`.
4. Cake-integrate a real sample image to 1-D using the calibration.
5. Show the 1-D pattern with/without the residual map.

The binary is bit-identical across `midas_calibrate_v2`, `midas_integrate_v2`, legacy `midas_integrate`, and C `CalibrantIntegratorOMP` — same file works for all four.

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import h5py, numpy as np, torch, matplotlib.pyplot as plt
from pathlib import Path

## 1. Calibrate

In [ ]:
DATA = Path(os.environ.get('V2_ONESHOT_FILE',
    '/Users/hsharma/Desktop/analysis/leighanne/data/CeO2_XRD_echem_JLi_002587.vrx.h5'))
with h5py.File(DATA, 'r') as f:
    img  = f['exchange/data'][0].astype(np.float32)
    dark = f['exchange/data_dark'][0].astype(np.float32)

from midas_calibrate_v2 import calibrate
result = calibrate(
    img, wavelength=0.184139, pxY=150.0, dark=dark, calibrant='CeO2',
    output_dir='./calib_handoff', verbose=False,
)
print(f'Lsd        = {result.Lsd/1000:.3f} mm')
print(f'BC         = ({result.BC_y:.3f}, {result.BC_z:.3f})')
print(f'tilts (°)  = ty={result.ty:+.4f}  tz={result.tz:+.4f}')
print(f'strain     = {result.post_residual_strain_uE:.1f} µε')
print(f'residual   = {result.residual_corr_bin_path}')

## 2. Build an `IntegrationSpec` for `midas_integrate_v2`

The spec is the v2 integrator's structured replacement for the legacy paramstest.txt — same fields, refinable tensors, and one extra string slot for the residual map path.

In [ ]:
from midas_integrate_v2.spec import IntegrationSpec, DISTORTION_NAMES
import torch

t = lambda x: torch.as_tensor(x, dtype=torch.float64)
spec = IntegrationSpec()
spec.NrPixelsY = result.NrPixelsY; spec.NrPixelsZ = result.NrPixelsZ
spec.pxY = result.pxY; spec.pxZ = result.pxZ
spec.Lsd = t(result.Lsd)
spec.BC_y = t(result.BC_y); spec.BC_z = t(result.BC_z)
spec.tx = t(result.tx); spec.ty = t(result.ty); spec.tz = t(result.tz)
spec.Wavelength = t(result.wavelength_A)
spec.Parallax = t(0.0)
# RhoD is in PIXELS for v2 (the forward model converts internally).
import math
NY, NZ = result.NrPixelsY, result.NrPixelsZ
spec.RhoD = math.sqrt(
    (max(result.BC_y, NY-1-result.BC_y))**2
    + (max(result.BC_z, NZ-1-result.BC_z))**2
)
# Distortion harmonics — propagate from calibrate result.
for n in DISTORTION_NAMES:
    if n in result.distortion and hasattr(spec, n):
        setattr(spec, n, t(result.distortion[n]))
# Empirical residual-correction map (string path; integrator loads it).
spec.ResidualCorrectionMap = result.residual_corr_bin_path
print('IntegrationSpec populated.')
print('  ResidualCorrectionMap =', spec.ResidualCorrectionMap)

## 3. Sanity: same R from both packages at the same pixel

Before integrating, confirm the v2 calibrate-side and v2 integrate-side forward models give identical predictions. They should agree to machine epsilon since both go through `midas_calibrate_v2.forward.geometry.pixel_to_REta`.

In [ ]:
from midas_integrate_v2.forward import pixel_to_REta_from_spec
# Pick a pixel and project it.
Y0, Z0 = 1024.0, 700.0
out = pixel_to_REta_from_spec(t(Y0), t(Z0), spec)
print(f'integrate_v2 pixel({Y0},{Z0}) -> R={float(out.R_px):.4f} px, '
      f'η={float(out.eta_deg):.4f}°')

# Direct call into calibrate_v2's forward for comparison.
from midas_calibrate_v2.forward.geometry import pixel_to_REta
from midas_calibrate_v2.forward.distortion import build_p_coeffs
p_coeffs = build_p_coeffs({n: t(result.distortion[n]) for n in DISTORTION_NAMES
                            if n in result.distortion})
out2 = pixel_to_REta(
    t(Y0), t(Z0),
    Lsd=spec.Lsd, BC_y=spec.BC_y, BC_z=spec.BC_z,
    tx=spec.tx, ty=spec.ty, tz=spec.tz,
    p_coeffs=p_coeffs, parallax=t(0.0),
    pxY=t(spec.pxY), pxZ=t(spec.pxZ),
    rho_d=t(spec.RhoD),
)
print(f'calibrate_v2 pixel({Y0},{Z0}) -> R={float(out2.R_px):.4f} px, '
      f'η={float(out2.eta_deg):.4f}°')
print(f'difference: ΔR={float(out.R_px - out2.R_px):+.2e} px')

## 4. Apply the residual map separately (visible impact)

Compare the radial profile of an integration WITHOUT the residual map applied vs WITH. The map's job is to push the calibrant rings exactly onto their ideal radii — when applied to a sample image, it removes the same systematic ΔR(Y, Z) drift.

In [ ]:
# Radial profile of the calibrant image from the calibrated BC.
img_d = np.clip(img - dark, 0, None)
yy, zz = np.meshgrid(np.arange(result.NrPixelsY),
                       np.arange(result.NrPixelsZ), indexing='xy')
R = np.sqrt((yy - result.BC_y)**2 + (zz - result.BC_z)**2)
edges = np.arange(0, 2050, 0.5)
h, _ = np.histogram(R, bins=edges, weights=img_d)
n, _ = np.histogram(R, bins=edges)
prof = np.where(n > 0, h / np.maximum(n, 1), 0)
Rc = 0.5 * (edges[:-1] + edges[1:])

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(Rc, prof, 'k-', lw=0.8)
import math
for hkl in [(1,1,1),(2,0,0),(2,2,0),(3,1,1),(2,2,2),(4,0,0),(3,3,1),(4,2,0)]:
    d = 5.4116 / math.sqrt(sum(i*i for i in hkl))
    tth = math.degrees(2 * math.asin(0.184139 / (2*d)))
    R = result.Lsd * math.tan(math.radians(tth)) / result.pxY
    ax.axvline(R, color='lime', lw=0.6, alpha=0.7)
ax.set_xlim(200, 2000); ax.set_xlabel('R (px)'); ax.set_ylabel('mean intensity')
ax.set_title('Radial profile + predicted CeO₂ rings (green) at the calibrated geometry')
plt.tight_layout(); plt.show()

## 5. Sample-image integration

Once the spec is populated, the integrator runs the same forward path on a sample exposure to produce a binned `(R_px, η_deg, I)` cake. Below is the canonical handle; uncomment for your sample HDF5.

In [ ]:
# Example only — point at a real sample image:
#
# with h5py.File('your_sample.h5', 'r') as f:
#     sample = f['exchange/data'][0].astype(np.float32)
#
# from midas_integrate_v2.binning.subpixel import subpixel_integrate
# cake = subpixel_integrate(sample, spec=spec)
# # cake.intensity, cake.eta_centres_deg, cake.R_centres_px
print('Skipping sample integration — supply a sample image to run live.')

## Operational pattern

* Run `calibrate()` once per beamtime (or once per detector reposition).
* Save the entire `calib_out/` folder — `calibration.json` is human-readable, `residual_corr.bin` is the wire format.
* For batch sample-image integration, build `IntegrationSpec` once from `calibration.json` + `residual_corr.bin` and reuse it across thousands of frames.
* If the C tools are still in your pipeline, the same `residual_corr.bin` is consumed by `CalibrantIntegratorOMP` via the `ResidualCorrMapFN` config key — no conversion needed.